In [1]:
import torch
import inspect
from pathlib import Path
from transformers import AutoModel, AutoVideoProcessor
from pprint import pprint

In [2]:
# ------------------------------------------------------------------
# MODELS
# ------------------------------------------------------------------

MODEL_IDS = [
    "facebook/vjepa2-vitl-fpc64-256",
    "facebook/vjepa2-vitl-fpc16-256-ssv2",
    "facebook/sparsh-vjepa-base",
    "facebook/sparsh-vjepa-small",
    "facebook/vjepa2-vitg-fpc64-384",
    "facebook/vjepa2-vitg-fpc64-256",
]

In [3]:
VIDEO_PATH = "/content/31October_2009_Saturday_tagesschau-7145.mp4"

In [4]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [5]:
# ------------------------------------------------------------------
# VIDEO LOADER
# ------------------------------------------------------------------

def load_video(video_path):
    """
    Uses processor's own video pipeline.
    Returns raw video input.
    """
    return video_path

In [6]:
# ------------------------------------------------------------------
# MODEL INSPECTION
# ------------------------------------------------------------------

def inspect_model(model, processor, model_id):

    print("\n" + "=" * 120)
    print(f"MODEL: {model_id}")
    print("=" * 120)

    print("\n[MODEL CLASS]")
    print(type(model))

    print("\n[PROCESSOR CLASS]")
    print(type(processor))

    print("\n[MODEL PRINT]")
    print(model)

    print("\n[CONFIG]")
    print(model.config.to_dict())

    print("\n[MODEL PARAMETERS]")

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(
        p.numel() for p in model.parameters()
        if p.requires_grad
    )

    print("total_params:", total_params)
    print("trainable_params:", trainable_params)

    print("\n[TOP LEVEL MODULES]")
    for name, module in model.named_children():
        print(name, "->", type(module).__name__)

    print("\n[FORWARD SIGNATURE]")
    print(inspect.signature(model.forward))

    print("\n[PROCESSOR ATTRIBUTES]")
    pprint(processor.__dict__)

In [7]:
# ------------------------------------------------------------------
# OUTPUT INSPECTION
# ------------------------------------------------------------------

def inspect_outputs(outputs):

    print("\n[OUTPUT TYPE]")
    print(type(outputs))

    if hasattr(outputs, "keys"):
        print("\n[OUTPUT KEYS]")
        print(list(outputs.keys()))

    print("\n[OUTPUT CONTENTS]")

    for attr in dir(outputs):

        if attr.startswith("_"):
            continue

        try:
            value = getattr(outputs, attr)

            if torch.is_tensor(value):
                print(
                    f"{attr:35s} "
                    f"shape={tuple(value.shape)} "
                    f"dtype={value.dtype}"
                )

            elif isinstance(value, (list, tuple)):

                if len(value) > 0 and torch.is_tensor(value[0]):
                    print(
                        f"{attr:35s} "
                        f"num_tensors={len(value)} "
                        f"first_shape={tuple(value[0].shape)}"
                    )
                else:
                    print(
                        f"{attr:35s} "
                        f"type={type(value)} "
                        f"len={len(value)}"
                    )

        except Exception:
            pass

In [8]:
# ------------------------------------------------------------------
# INPUT INSPECTION
# ------------------------------------------------------------------

def inspect_inputs(inputs):

    print("\n[MODEL INPUTS]")

    for k, v in inputs.items():

        if torch.is_tensor(v):
            print(
                f"{k:25s} "
                f"shape={tuple(v.shape)} "
                f"dtype={v.dtype}"
            )
        else:
            print(k, type(v))

In [34]:
model_id = "facebook/vjepa2-vitl-fpc64-256"
try:

        print("\n\n")
        print("#" * 120)
        print(f"LOADING {model_id}")
        print("#" * 120)

        processor = AutoVideoProcessor.from_pretrained(
            model_id,
            trust_remote_code=True,
        )

        model = AutoModel.from_pretrained(
            model_id,
            trust_remote_code=True,
        )

        model.eval()
        model.to(DEVICE)

        inspect_model(
            model=model,
            processor=processor,
            model_id=model_id,
        )

        # ----------------------------------------------------------
        # Prepare video
        # ----------------------------------------------------------

        video = load_video(VIDEO_PATH)

        inputs = processor(
            videos=video,
            return_tensors="pt",
        )

        inputs = {
            k: v.to(DEVICE)
            if torch.is_tensor(v)
            else v
            for k, v in inputs.items()
        }

        inspect_inputs(inputs)

        # ----------------------------------------------------------
        # Forward
        # ----------------------------------------------------------

        with torch.no_grad():

            outputs = model(
                **inputs,
                output_hidden_states=True,
                output_attentions=True,
            )

        inspect_outputs(outputs)

        # ----------------------------------------------------------
        # Hidden states
        # ----------------------------------------------------------

        if hasattr(outputs, "hidden_states"):

            print("\n[HIDDEN STATES]")

            for idx, hs in enumerate(outputs.hidden_states):

                print(
                    f"layer={idx:<3d} "
                    f"shape={tuple(hs.shape)} "
                    f"dtype={hs.dtype}"
                )

        # ----------------------------------------------------------
        # Attentions
        # ----------------------------------------------------------

        if hasattr(outputs, "attentions"):

            print("\n[ATTENTIONS]")

            for idx, attn in enumerate(outputs.attentions):

                print(
                    f"layer={idx:<3d} "
                    f"shape={tuple(attn.shape)} "
                    f"dtype={attn.dtype}"
                )

        print("\nMODEL COMPLETED SUCCESSFULLY")

except Exception as e:

        print("\nFAILED:", model_id)
        print(type(e).__name__)
        print(str(e))




########################################################################################################################
LOADING facebook/vjepa2-vitl-fpc64-256
########################################################################################################################


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]


MODEL: facebook/vjepa2-vitl-fpc64-256

[MODEL CLASS]
<class 'transformers.models.vjepa2.modeling_vjepa2.VJEPA2Model'>

[PROCESSOR CLASS]
<class 'transformers.models.vjepa2.video_processing_vjepa2.VJEPA2VideoProcessor'>

[MODEL PRINT]
VJEPA2Model(
  (encoder): VJEPA2Encoder(
    (embeddings): VJEPA2Embeddings(
      (patch_embeddings): VJEPA2PatchEmbeddings3D(
        (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
    )
    (layer): ModuleList(
      (0-23): 24 x VJEPA2Layer(
        (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
        (attention): VJEPA2RopeAttention(
          (query): Linear(in_features=1024, out_features=1024, bias=True)
          (key): Linear(in_features=1024, out_features=1024, bias=True)
          (value): Linear(in_features=1024, out_features=1024, bias=True)
          (proj): Linear(in_features=1024, out_features=1024, bias=True)
          (dropout): Dropout(p=0.0, inplace=False)
        )
        (drop_path): 

In [35]:
print(model.config)
print("output_attentions:", getattr(model.config, "output_attentions"))

VJEPA2Config {
  "architectures": [
    "VJEPA2Model"
  ],
  "attention_dropout": 0.0,
  "attention_probs_dropout_prob": 0.0,
  "crop_size": 256,
  "drop_path_rate": 0.0,
  "dtype": "float32",
  "frames_per_clip": 64,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 1024,
  "image_size": 256,
  "in_chans": 3,
  "initializer_range": 0.02,
  "layer_norm_eps": 1e-06,
  "mlp_ratio": 4,
  "model_type": "vjepa2",
  "num_attention_heads": 16,
  "num_hidden_layers": 24,
  "num_pooler_layers": 3,
  "patch_size": 16,
  "pred_hidden_size": 384,
  "pred_mlp_ratio": 4.0,
  "pred_num_attention_heads": 12,
  "pred_num_hidden_layers": 12,
  "pred_num_mask_tokens": 10,
  "pred_zero_init_mask_tokens": true,
  "qkv_bias": true,
  "transformers_version": "5.10.2",
  "tubelet_size": 2,
  "use_SiLU": false,
  "wide_SiLU": true
}

output_attentions: False


In [36]:
model_id = "facebook/vjepa2-vitl-fpc16-256-ssv2"
try:

        print("\n\n")
        print("#" * 120)
        print(f"LOADING {model_id}")
        print("#" * 120)

        processor = AutoVideoProcessor.from_pretrained(
            model_id,
            trust_remote_code=True,
        )

        model = AutoModel.from_pretrained(
            model_id,
            trust_remote_code=True,
        )

        model.eval()
        model.to(DEVICE)

        inspect_model(
            model=model,
            processor=processor,
            model_id=model_id,
        )

        # ----------------------------------------------------------
        # Prepare video
        # ----------------------------------------------------------

        video = load_video(VIDEO_PATH)

        inputs = processor(
            videos=video,
            return_tensors="pt",
        )

        inputs = {
            k: v.to(DEVICE)
            if torch.is_tensor(v)
            else v
            for k, v in inputs.items()
        }

        inspect_inputs(inputs)

        # ----------------------------------------------------------
        # Forward
        # ----------------------------------------------------------

        with torch.no_grad():

            outputs = model(
                **inputs,
                output_hidden_states=True,
                output_attentions=True,
            )

        inspect_outputs(outputs)

        # ----------------------------------------------------------
        # Hidden states
        # ----------------------------------------------------------

        if hasattr(outputs, "hidden_states"):

            print("\n[HIDDEN STATES]")

            for idx, hs in enumerate(outputs.hidden_states):

                print(
                    f"layer={idx:<3d} "
                    f"shape={tuple(hs.shape)} "
                    f"dtype={hs.dtype}"
                )

        # ----------------------------------------------------------
        # Attentions
        # ----------------------------------------------------------

        if hasattr(outputs, "attentions"):

            print("\n[ATTENTIONS]")

            for idx, attn in enumerate(outputs.attentions):

                print(
                    f"layer={idx:<3d} "
                    f"shape={tuple(attn.shape)} "
                    f"dtype={attn.dtype}"
                )

        print("\nMODEL COMPLETED SUCCESSFULLY")

except Exception as e:

        print("\nFAILED:", model_id)
        print(type(e).__name__)
        print(str(e))




########################################################################################################################
LOADING facebook/vjepa2-vitl-fpc16-256-ssv2
########################################################################################################################


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

[transformers] VJEPA2Model LOAD REPORT from: facebook/vjepa2-vitl-fpc16-256-ssv2
Key                                                              | Status     |  | 
-----------------------------------------------------------------+------------+--+-
pooler.self_attention_layers.{0, 1, 2}.layer_norm2.bias          | UNEXPECTED |  | 
pooler.self_attention_layers.{0, 1, 2}.mlp.fc2.bias              | UNEXPECTED |  | 
pooler.self_attention_layers.{0, 1, 2}.self_attn.out_proj.weight | UNEXPECTED |  | 
pooler.query_tokens                                              | UNEXPECTED |  | 
pooler.self_attention_layers.{0, 1, 2}.self_attn.v_proj.weight   | UNEXPECTED |  | 
pooler.self_attention_layers.{0, 1, 2}.self_attn.out_proj.bias   | UNEXPECTED |  | 
pooler.self_attention_layers.{0, 1, 2}.self_attn.q_proj.bias     | UNEXPECTED |  | 
pooler.self_attention_layers.{0, 1, 2}.layer_norm2.weight        | UNEXPECTED |  | 
pooler.cross_attention_layer.cross_attn.v_proj.bias              | UNEXPECTED |


MODEL: facebook/vjepa2-vitl-fpc16-256-ssv2

[MODEL CLASS]
<class 'transformers.models.vjepa2.modeling_vjepa2.VJEPA2Model'>

[PROCESSOR CLASS]
<class 'transformers.models.vjepa2.video_processing_vjepa2.VJEPA2VideoProcessor'>

[MODEL PRINT]
VJEPA2Model(
  (encoder): VJEPA2Encoder(
    (embeddings): VJEPA2Embeddings(
      (patch_embeddings): VJEPA2PatchEmbeddings3D(
        (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
    )
    (layer): ModuleList(
      (0-23): 24 x VJEPA2Layer(
        (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
        (attention): VJEPA2RopeAttention(
          (query): Linear(in_features=1024, out_features=1024, bias=True)
          (key): Linear(in_features=1024, out_features=1024, bias=True)
          (value): Linear(in_features=1024, out_features=1024, bias=True)
          (proj): Linear(in_features=1024, out_features=1024, bias=True)
          (dropout): Dropout(p=0.0, inplace=False)
        )
        (drop_pa

In [37]:
model_id = "facebook/sparsh-vjepa-base"

try:

        print("\n\n")
        print("#" * 120)
        print(f"LOADING {model_id}")
        print("#" * 120)

        processor = AutoVideoProcessor.from_pretrained(
            model_id,
            trust_remote_code=True,
        )

        model = AutoModel.from_pretrained(
            model_id,
            trust_remote_code=True,
        )

        model.eval()
        model.to(DEVICE)

        inspect_model(
            model=model,
            processor=processor,
            model_id=model_id,
        )

        # ----------------------------------------------------------
        # Prepare video
        # ----------------------------------------------------------

        video = load_video(VIDEO_PATH)

        inputs = processor(
            videos=video,
            return_tensors="pt",
        )

        inputs = {
            k: v.to(DEVICE)
            if torch.is_tensor(v)
            else v
            for k, v in inputs.items()
        }

        inspect_inputs(inputs)

        # ----------------------------------------------------------
        # Forward
        # ----------------------------------------------------------

        with torch.no_grad():

            outputs = model(
                **inputs,
                output_hidden_states=True,
                output_attentions=True,
            )

        inspect_outputs(outputs)

        # ----------------------------------------------------------
        # Hidden states
        # ----------------------------------------------------------

        if hasattr(outputs, "hidden_states"):

            print("\n[HIDDEN STATES]")

            for idx, hs in enumerate(outputs.hidden_states):

                print(
                    f"layer={idx:<3d} "
                    f"shape={tuple(hs.shape)} "
                    f"dtype={hs.dtype}"
                )

        # ----------------------------------------------------------
        # Attentions
        # ----------------------------------------------------------

        if hasattr(outputs, "attentions"):

            print("\n[ATTENTIONS]")

            for idx, attn in enumerate(outputs.attentions):

                print(
                    f"layer={idx:<3d} "
                    f"shape={tuple(attn.shape)} "
                    f"dtype={attn.dtype}"
                )

        print("\nMODEL COMPLETED SUCCESSFULLY")

except Exception as e:

        print("\nFAILED:", model_id)
        print(type(e).__name__)
        print(str(e))




########################################################################################################################
LOADING facebook/sparsh-vjepa-base
########################################################################################################################

FAILED: facebook/sparsh-vjepa-base
OSError
Can't load video processor for 'facebook/sparsh-vjepa-base'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'facebook/sparsh-vjepa-base' is the correct path to a directory containing a video_preprocessor_config.json file


In [38]:
model_id = "facebook/sparsh-vjepa-small"

try:

        print("\n\n")
        print("#" * 120)
        print(f"LOADING {model_id}")
        print("#" * 120)

        processor = AutoVideoProcessor.from_pretrained(
            model_id,
            trust_remote_code=True,
        )

        model = AutoModel.from_pretrained(
            model_id,
            trust_remote_code=True,
        )

        model.eval()
        model.to(DEVICE)

        inspect_model(
            model=model,
            processor=processor,
            model_id=model_id,
        )

        # ----------------------------------------------------------
        # Prepare video
        # ----------------------------------------------------------

        video = load_video(VIDEO_PATH)

        inputs = processor(
            videos=video,
            return_tensors="pt",
        )

        inputs = {
            k: v.to(DEVICE)
            if torch.is_tensor(v)
            else v
            for k, v in inputs.items()
        }

        inspect_inputs(inputs)

        # ----------------------------------------------------------
        # Forward
        # ----------------------------------------------------------

        with torch.no_grad():

            outputs = model(
                **inputs,
                output_hidden_states=True,
                output_attentions=True,
            )

        inspect_outputs(outputs)

        # ----------------------------------------------------------
        # Hidden states
        # ----------------------------------------------------------

        if hasattr(outputs, "hidden_states"):

            print("\n[HIDDEN STATES]")

            for idx, hs in enumerate(outputs.hidden_states):

                print(
                    f"layer={idx:<3d} "
                    f"shape={tuple(hs.shape)} "
                    f"dtype={hs.dtype}"
                )

        # ----------------------------------------------------------
        # Attentions
        # ----------------------------------------------------------

        if hasattr(outputs, "attentions"):

            print("\n[ATTENTIONS]")

            for idx, attn in enumerate(outputs.attentions):

                print(
                    f"layer={idx:<3d} "
                    f"shape={tuple(attn.shape)} "
                    f"dtype={attn.dtype}"
                )

        print("\nMODEL COMPLETED SUCCESSFULLY")

except Exception as e:

        print("\nFAILED:", model_id)
        print(type(e).__name__)
        print(str(e))




########################################################################################################################
LOADING facebook/sparsh-vjepa-small
########################################################################################################################

FAILED: facebook/sparsh-vjepa-small
OSError
Can't load video processor for 'facebook/sparsh-vjepa-small'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'facebook/sparsh-vjepa-small' is the correct path to a directory containing a video_preprocessor_config.json file


In [9]:
model_id = "facebook/vjepa2-vitg-fpc64-384"

try:

        print("\n\n")
        print("#" * 120)
        print(f"LOADING {model_id}")
        print("#" * 120)

        processor = AutoVideoProcessor.from_pretrained(
            model_id,
            trust_remote_code=True,
        )

        model = AutoModel.from_pretrained(
            model_id,
            trust_remote_code=True,
        )

        model.eval()
        model.to(DEVICE)

        inspect_model(
            model=model,
            processor=processor,
            model_id=model_id,
        )

        # ----------------------------------------------------------
        # Prepare video
        # ----------------------------------------------------------

        video = load_video(VIDEO_PATH)

        inputs = processor(
            videos=video,
            return_tensors="pt",
        )

        inputs = {
            k: v.to(DEVICE)
            if torch.is_tensor(v)
            else v
            for k, v in inputs.items()
        }

        inspect_inputs(inputs)

        # ----------------------------------------------------------
        # Forward
        # ----------------------------------------------------------

        with torch.no_grad():

            outputs = model(
                **inputs,
                output_hidden_states=True,
                output_attentions=True,
            )

        inspect_outputs(outputs)

        # ----------------------------------------------------------
        # Hidden states
        # ----------------------------------------------------------

        if hasattr(outputs, "hidden_states"):

            print("\n[HIDDEN STATES]")

            for idx, hs in enumerate(outputs.hidden_states):

                print(
                    f"layer={idx:<3d} "
                    f"shape={tuple(hs.shape)} "
                    f"dtype={hs.dtype}"
                )

        # ----------------------------------------------------------
        # Attentions
        # ----------------------------------------------------------

        if hasattr(outputs, "attentions"):

            print("\n[ATTENTIONS]")

            for idx, attn in enumerate(outputs.attentions):

                print(
                    f"layer={idx:<3d} "
                    f"shape={tuple(attn.shape)} "
                    f"dtype={attn.dtype}"
                )

        print("\nMODEL COMPLETED SUCCESSFULLY")

except Exception as e:

        print("\nFAILED:", model_id)
        print(type(e).__name__)
        print(str(e))




########################################################################################################################
LOADING facebook/vjepa2-vitg-fpc64-384
########################################################################################################################


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


video_preprocessor_config.json:   0%|          | 0.00/1.30k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/780 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/843 [00:00<?, ?it/s]


MODEL: facebook/vjepa2-vitg-fpc64-384

[MODEL CLASS]
<class 'transformers.models.vjepa2.modeling_vjepa2.VJEPA2Model'>

[PROCESSOR CLASS]
<class 'transformers.models.vjepa2.video_processing_vjepa2.VJEPA2VideoProcessor'>

[MODEL PRINT]
VJEPA2Model(
  (encoder): VJEPA2Encoder(
    (embeddings): VJEPA2Embeddings(
      (patch_embeddings): VJEPA2PatchEmbeddings3D(
        (proj): Conv3d(3, 1408, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
    )
    (layer): ModuleList(
      (0-39): 40 x VJEPA2Layer(
        (norm1): LayerNorm((1408,), eps=1e-06, elementwise_affine=True)
        (attention): VJEPA2RopeAttention(
          (query): Linear(in_features=1408, out_features=1408, bias=True)
          (key): Linear(in_features=1408, out_features=1408, bias=True)
          (value): Linear(in_features=1408, out_features=1408, bias=True)
          (proj): Linear(in_features=1408, out_features=1408, bias=True)
          (dropout): Dropout(p=0.0, inplace=False)
        )
        (drop_path): 

In [40]:
model_id = "facebook/vjepa2-vitg-fpc64-256"

try:

        print("\n\n")
        print("#" * 120)
        print(f"LOADING {model_id}")
        print("#" * 120)

        processor = AutoVideoProcessor.from_pretrained(
            model_id,
            trust_remote_code=True,
        )

        model = AutoModel.from_pretrained(
            model_id,
            trust_remote_code=True,
        )

        model.eval()
        model.to(DEVICE)

        inspect_model(
            model=model,
            processor=processor,
            model_id=model_id,
        )

        # ----------------------------------------------------------
        # Prepare video
        # ----------------------------------------------------------

        video = load_video(VIDEO_PATH)

        inputs = processor(
            videos=video,
            return_tensors="pt",
        )

        inputs = {
            k: v.to(DEVICE)
            if torch.is_tensor(v)
            else v
            for k, v in inputs.items()
        }

        inspect_inputs(inputs)

        # ----------------------------------------------------------
        # Forward
        # ----------------------------------------------------------

        with torch.no_grad():

            outputs = model(
                **inputs,
                output_hidden_states=True,
                output_attentions=True,
            )

        inspect_outputs(outputs)

        # ----------------------------------------------------------
        # Hidden states
        # ----------------------------------------------------------

        if hasattr(outputs, "hidden_states"):

            print("\n[HIDDEN STATES]")

            for idx, hs in enumerate(outputs.hidden_states):

                print(
                    f"layer={idx:<3d} "
                    f"shape={tuple(hs.shape)} "
                    f"dtype={hs.dtype}"
                )

        # ----------------------------------------------------------
        # Attentions
        # ----------------------------------------------------------

        if hasattr(outputs, "attentions"):

            print("\n[ATTENTIONS]")

            for idx, attn in enumerate(outputs.attentions):

                print(
                    f"layer={idx:<3d} "
                    f"shape={tuple(attn.shape)} "
                    f"dtype={attn.dtype}"
                )

        print("\nMODEL COMPLETED SUCCESSFULLY")

except Exception as e:

        print("\nFAILED:", model_id)
        print(type(e).__name__)
        print(str(e))




########################################################################################################################
LOADING facebook/vjepa2-vitg-fpc64-256
########################################################################################################################


Loading weights:   0%|          | 0/843 [00:00<?, ?it/s]


MODEL: facebook/vjepa2-vitg-fpc64-256

[MODEL CLASS]
<class 'transformers.models.vjepa2.modeling_vjepa2.VJEPA2Model'>

[PROCESSOR CLASS]
<class 'transformers.models.vjepa2.video_processing_vjepa2.VJEPA2VideoProcessor'>

[MODEL PRINT]
VJEPA2Model(
  (encoder): VJEPA2Encoder(
    (embeddings): VJEPA2Embeddings(
      (patch_embeddings): VJEPA2PatchEmbeddings3D(
        (proj): Conv3d(3, 1408, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
    )
    (layer): ModuleList(
      (0-39): 40 x VJEPA2Layer(
        (norm1): LayerNorm((1408,), eps=1e-06, elementwise_affine=True)
        (attention): VJEPA2RopeAttention(
          (query): Linear(in_features=1408, out_features=1408, bias=True)
          (key): Linear(in_features=1408, out_features=1408, bias=True)
          (value): Linear(in_features=1408, out_features=1408, bias=True)
          (proj): Linear(in_features=1408, out_features=1408, bias=True)
          (dropout): Dropout(p=0.0, inplace=False)
        )
        (drop_path): 